# HW03 — Model Serving & Deployment

Your model is trained and tracked in MLflow. A model that only runs in a notebook is not useful in production.

In this homework you will:

- verify that your trained model is reproducible and ready for serving
- wrap it in a FastAPI service with proper input validation and batch support
- measure the performance difference between single and batch prediction
- package the service in Docker with a size-optimized image
- write Kubernetes manifests to deploy the service

## Submission discipline

This is individual work.

Work locally. Push to GitHub. Do not SSH into the server.

Do not commit `.env`, `.venv/`, passwords, tokens, or notebook checkpoints.
Do not hardcode passwords anywhere in your code.

## Useful references

- MLflow model loading: https://mlflow.org/docs/latest/python_api/mlflow.sklearn.html
- FastAPI: https://fastapi.tiangolo.com/
- FastAPI lifespan: https://fastapi.tiangolo.com/advanced/events/
- Pydantic v2: https://docs.pydantic.dev/latest/
- Dockerfile reference: https://docs.docker.com/reference/dockerfile/
- Docker multi-stage builds: https://docs.docker.com/build/building/multi-stage/
- Kubernetes Deployments: https://kubernetes.io/docs/concepts/workloads/controllers/deployment/
- Kubernetes Services: https://kubernetes.io/docs/concepts/services-networking/service/

## What to avoid

- Loading the model inside the request handler. Load once at startup.
- Hardcoded passwords in any source file.
- A Docker image that bakes in the model file. Pull from MLflow at startup.
- Returning raw numpy types from the API. JSON needs native Python types.
- Skipping the batch vs single benchmark. The numbers tell a story.

In [6]:
import os
import re
import time
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient

PROJECT = Path.cwd()
for path in ['src/airbnb_serving', 'k8s', 'reports', 'screenshots']:
    (PROJECT / path).mkdir(parents=True, exist_ok=True)
(PROJECT / 'src/airbnb_serving/__init__.py').write_text('__version__ = "0.1.0"\n')

STUDENT_ID = os.getenv('QBC12_STUDENT_ID', '') or input('GitHub username / student id: ').strip()
safe_student = re.sub(r'[^a-zA-Z0-9_]', '_', STUDENT_ID.lower())
EXPERIMENT_NAME = f'qbc12_hw02_{safe_student}'

print('PROJECT:', PROJECT)
print('EXPERIMENT_NAME:', EXPERIMENT_NAME)

PROJECT: /home/mehrad/Desktop/HW03/project1
EXPERIMENT_NAME: qbc12_hw02_student_mehrad_rafiei_tabatabaei


---
## Part 1 — Model Reproducibility Check

Before serving a model, you must verify it produces exactly the same output as it did during training.

This is called a **reproducibility check** and it catches silent bugs like:
- preprocessing mismatch between training and serving
- wrong model version loaded
- feature column order changed

### 1.1 Connect to MLflow and load your best model

In [7]:
MLFLOW_TRACKING_URI = os.getenv('MLFLOW_TRACKING_URI', 'http://185.50.38.163:33014')
MLFLOW_USERNAME = os.getenv('MLFLOW_TRACKING_USERNAME', '') or input('MLflow username: ').strip()
MLFLOW_PASSWORD = os.getenv('MLFLOW_TRACKING_PASSWORD', '') or input('MLflow password: ').strip()

os.environ['MLFLOW_TRACKING_USERNAME'] = MLFLOW_USERNAME
os.environ['MLFLOW_TRACKING_PASSWORD'] = MLFLOW_PASSWORD

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = MlflowClient()

experiment = client.get_experiment_by_name(EXPERIMENT_NAME)
if experiment is None:
    raise ValueError(f'Experiment not found: {EXPERIMENT_NAME}. Complete HW02 first.')

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="tags.leakage_status = 'clean' and tags.selected_for_serving = 'true'",
    order_by=['metrics.f1 DESC'],
)

if runs.empty:
    raise ValueError(
        'No run tagged selected_for_serving=true found. '
        'Go to MLflow UI, find your best clean run, and add the tag.'
    )

BEST_RUN_ID = runs.iloc[0]['run_id']
MODEL_URI = f'runs:/{BEST_RUN_ID}/model'

print('Best run ID  :', BEST_RUN_ID)
print('Model URI    :', MODEL_URI)
print('Run name     :', runs.iloc[0].get('tags.mlflow.runName'))
print('F1 score     :', runs.iloc[0].get('metrics.f1'))

Best run ID  : 258390aa321c4947a44002e31c3d13a7
Model URI    : runs:/258390aa321c4947a44002e31c3d13a7/model
Run name     : v5_random_forest
F1 score     : 0.9884411121524523


In [8]:
model = mlflow.sklearn.load_model(MODEL_URI)
print('Model type:', type(model))
print('Model steps:', list(model.named_steps.keys()) if hasattr(model, 'named_steps') else 'not a pipeline')

Model type: <class 'sklearn.pipeline.Pipeline'>
Model steps: ['preprocessor', 'classifier']


### 1.2 Load your HW01 feature dataset

You will use a small sample from your HW01 feature parquet file to verify reproducibility.

In [9]:
FEATURE_COLS = [
    'instant_bookable', 'accommodates', 'bathrooms', 'bedrooms', 'beds',
    'listing_price', 'minimum_nights', 'maximum_nights', 'is_superhost',
    'host_listing_count', 'neighbourhood_name',
    'total_reviews_before_cutoff', 'unique_reviewers_before_cutoff',
    'avg_comment_len_before_cutoff', 'max_comment_len_before_cutoff',
    'days_since_last_review', 'available_days_last_90d', 'available_rate_last_90d',
    'avg_minimum_nights_calendar_last_90d', 'avg_maximum_nights_calendar_last_90d',
    'available_days_last_30d', 'available_rate_last_30d',
    'avg_minimum_nights_calendar_last_30d', 'avg_maximum_nights_calendar_last_30d',
]
TARGET_COL = 'high_demand_proxy'

# Load your HW01 parquet file
# Adjust the path if needed
parquet_path = list(Path('data/features').glob('*.parquet'))
if not parquet_path:
    raise FileNotFoundError('HW01 feature parquet not found. Run HW01 ETL first.')

df = pd.read_parquet(parquet_path[0])
print('Dataset shape:', df.shape)
df[FEATURE_COLS + [TARGET_COL]].head(3)

Dataset shape: (10480, 31)


,instant_bookable,accommodates,bathrooms,bedrooms,beds,listing_price,minimum_nights,maximum_nights,is_superhost,host_listing_count,...,days_since_last_review,available_days_last_90d,available_rate_last_90d,avg_minimum_nights_calendar_last_90d,avg_maximum_nights_calendar_last_90d,available_days_last_30d,available_rate_last_30d,avg_minimum_nights_calendar_last_30d,avg_maximum_nights_calendar_last_30d,high_demand_proxy
0,False,2,1.5,1.0,1.0,132.0,3,356,True,1,...,5328.0,0,0.000000,3.0,30.0,0,0.000000,3.0,30.0,1
1,False,2,1.0,1.0,1.0,89.0,2,730,True,2,...,5833.0,46,0.505495,2.0,730.0,14,0.466667,2.0,730.0,0
2,False,2,1.0,1.0,1.0,61.0,2,730,True,2,...,5627.0,43,0.472527,2.0,730.0,16,0.533333,2.0,730.0,1


### 1.3 Reproducibility check

**TODO 1.3**

Take a sample of 50 rows from the dataset.

Run `model.predict()` on those rows **twice** and verify the results are identical.

Then compare the predictions against the `high_demand_proxy` column and print:
- how many predictions match the training label
- the accuracy on this sample

If both runs produce identical output, print `Reproducibility check passed.`
If they differ, raise a `ValueError`.

In [10]:
sample = df[FEATURE_COLS + [TARGET_COL]].sample(50, random_state=42)
X_sample = sample[FEATURE_COLS]
y_sample = sample[TARGET_COL]

predictions_1 = model.predict(X_sample)
predictions_2 = model.predict(X_sample)

if not np.array_equal(predictions_1, predictions_2):
    raise ValueError('Reproducibility check failed: predictions differ between runs.')

matches = int((predictions_1 == y_sample.to_numpy()).sum())
accuracy = matches / len(y_sample)

print(f'Matches: {matches}/{len(y_sample)}')
print(f'Accuracy: {accuracy:.3f}')
print('Reproducibility check passed.')

Matches: 49/50
Accuracy: 0.980
Reproducibility check passed.


---
## Part 2 — FastAPI Service

A REST API is the standard way to expose an ML model to other systems.

You will build a FastAPI app with two prediction endpoints:
- `POST /predict` — single listing prediction
- `POST /predict/batch` — multiple listings in one request

Then you will measure how much faster batch is compared to calling single predict in a loop.

### 2.1 Input and output schemas

In [11]:
schema_source = (PROJECT / 'src/airbnb_serving/schema.py').read_text()
assert 'BaseModel' in schema_source
print('schema.py is complete.')

schema.py is complete.


### 2.2 Prediction logic

In [12]:
predictor_source = (PROJECT / 'src/airbnb_serving/predictor.py').read_text()
assert 'def predict_single' in predictor_source
assert 'def predict_batch' in predictor_source
print('predictor.py is complete.')

predictor.py is complete.


### 2.3 FastAPI app

In [13]:
app_source = (PROJECT / 'src/airbnb_serving/app.py').read_text()
assert 'lifespan' in app_source
assert '/health' in app_source
assert '/predict' in app_source
assert 'batch' in app_source
print('app.py is complete.')

app.py is complete.


### 2.4 Package metadata

In [14]:
assert (PROJECT / 'pyproject.toml').exists()
assert (PROJECT / 'requirements.txt').exists()
assert 'airbnb-serving' in (PROJECT / 'pyproject.toml').read_text()
print('Package metadata is complete.')

Package metadata is complete.


### 2.5 Local install and smoke test

Install the package and manually start the server in a terminal before running the test cell below.

```bash
pip install -e .

MODEL_RUN_ID=<your_run_id> \
MLFLOW_TRACKING_URI=http://185.50.38.163:33014 \
MLFLOW_TRACKING_USERNAME=<user> \
MLFLOW_TRACKING_PASSWORD=<pass> \
uvicorn airbnb_serving.app:app --host 0.0.0.0 --port 8000
```

In [15]:
import sys
!{sys.executable} -m pip install -q -e .

In [16]:
import subprocess, time, signal, os

server_proc = subprocess.Popen(
    ['uvicorn', 'airbnb_serving.app:app', '--host', '0.0.0.0', '--port', '12345'],
    env={**os.environ, 'MODEL_RUN_ID': BEST_RUN_ID},
)
time.sleep(15)  # wait for model to load from MLflow
print('Server started, PID:', server_proc.pid)

INFO:     Started server process [902546]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:12345 (Press CTRL+C to quit)


Server started, PID: 902546


In [17]:
import requests

BASE_URL = 'http://localhost:12345'

health = requests.get(f'{BASE_URL}/health')
assert health.status_code == 200, f'Health check failed: {health.text}'
print('Health:', health.json())

sample_payload = {
    'instant_bookable': True,
    'accommodates': 2,
    'bathrooms': 1.0,
    'bedrooms': 1.0,
    'beds': 1.0,
    'listing_price': 150.0,
    'minimum_nights': 2,
    'maximum_nights': 365,
    'is_superhost': False,
    'host_listing_count': 1,
    'neighbourhood_name': 'Centrum-West',
    'total_reviews_before_cutoff': 10.0,
    'unique_reviewers_before_cutoff': 9.0,
    'avg_comment_len_before_cutoff': 120.0,
    'max_comment_len_before_cutoff': 300.0,
    'days_since_last_review': 30.0,
    'available_days_last_90d': 45,
    'available_rate_last_90d': 0.5,
    'avg_minimum_nights_calendar_last_90d': 2.0,
    'avg_maximum_nights_calendar_last_90d': 365.0,
    'available_days_last_30d': 15,
    'available_rate_last_30d': 0.5,
    'avg_minimum_nights_calendar_last_30d': 2.0,
    'avg_maximum_nights_calendar_last_30d': 365.0,
}

resp = requests.post(f'{BASE_URL}/predict', json=sample_payload)
assert resp.status_code == 200, f'Single predict failed: {resp.text}'
print('Single predict:', resp.json())

print('Local smoke test passed.')

INFO:     127.0.0.1:43322 - "GET /health HTTP/1.1" 200 OK
Health: {'status': 'ok', 'model_run_id': '258390aa321c4947a44002e31c3d13a7'}
INFO:     127.0.0.1:43332 - "POST /predict HTTP/1.1" 200 OK
Single predict: {'listing_id': None, 'prediction': 0, 'probability_high_demand': 0.06044310575590878, 'model_run_id': '258390aa321c4947a44002e31c3d13a7'}
Local smoke test passed.


### 2.6 Batch vs single benchmark

**TODO 2.6**

Take 100 rows from your feature dataset.

Measure:
1. Time to call `POST /predict` 100 times in a loop (single)
2. Time to call `POST /predict/batch` once with all 100 rows (batch)

Print a comparison table with total time and time per prediction for each approach.

Then answer: why is batch faster? Write your answer as a comment in the cell.

In [18]:
import math

def clean_for_json(row: dict) -> dict:
    return {
        k: None if isinstance(v, float) and math.isnan(v) else v
        for k, v in row.items()
    }

BENCHMARK_SIZE = 100
benchmark_rows = [
    clean_for_json(row)
    for row in df[FEATURE_COLS].head(BENCHMARK_SIZE).to_dict(orient='records')
]

start = time.perf_counter()
for row in benchmark_rows:
    resp = requests.post(f'{BASE_URL}/predict', json=row)
    assert resp.status_code == 200, resp.text
single_total = time.perf_counter() - start

start = time.perf_counter()
resp = requests.post(f'{BASE_URL}/predict/batch', json=benchmark_rows)
assert resp.status_code == 200, resp.text
batch_total = time.perf_counter() - start

print('Method        | Total (s) | Per prediction (ms)')
print(f'single loop   | {single_total:9.3f} | {single_total / BENCHMARK_SIZE * 1000:19.2f}')
print(f'batch         | {batch_total:9.3f} | {batch_total / BENCHMARK_SIZE * 1000:19.2f}')
print(f'Speedup: {single_total / batch_total:.2f}x')

# Batch is faster because it pays HTTP, validation, DataFrame construction,
# and model pipeline overhead once instead of once per row.

INFO:     127.0.0.1:45040 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:45052 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:45064 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:45068 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:45078 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:45084 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:45086 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:45098 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:45114 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:45116 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:45118 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:45132 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:45136 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:45138 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:45146 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:45150 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:45166 - "POST /predi

In [19]:
server_proc.terminate()
print('Server stopped.')

Server stopped.


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [902546]


---
## Part 3 — Docker

You will build two Docker images and compare their sizes.

This teaches you that image size is not free — it affects pull time, storage cost, and attack surface.

### 3.1 Naive Dockerfile

In [26]:
assert (PROJECT / 'Dockerfile.naive').exists()
print('Dockerfile.naive is complete.')

Dockerfile.naive is complete.


### 3.2 Optimized Dockerfile

A multi-stage build separates the build environment from the runtime environment.

Stage 1 (builder): install everything, build the package.
Stage 2 (runtime): copy only what is needed to run, nothing else.

In [27]:
assert (PROJECT / 'Dockerfile').exists()
assert (PROJECT / '.dockerignore').exists()
print('Optimized Dockerfile and .dockerignore are complete.')

Optimized Dockerfile and .dockerignore are complete.


### 3.3 Build and compare image sizes

In [28]:
!docker build -f Dockerfile.naive -t qbc12-airbnb-serving:naive .
!docker build -f Dockerfile -t qbc12-airbnb-serving:optimized .

ERROR: permission denied while trying to connect to the docker API at unix:///var/run/docker.sock
ERROR: permission denied while trying to connect to the docker API at unix:///var/run/docker.sock


In [29]:
# Run this cell after building both images.
# It compares image sizes and saves the result to reports/.

import subprocess, json

result = subprocess.run(
    ['docker', 'images', '--format', '{{json .}}'],
    capture_output=True, text=True
)

images = [json.loads(line) for line in result.stdout.strip().split('\n') if line]
serving_images = [
    img for img in images
    if img.get('Repository') == 'qbc12-airbnb-serving'
]

size_df = pd.DataFrame(serving_images)[['Repository', 'Tag', 'Size']]
print(size_df.to_string(index=False))

report_lines = [
    '# HW03 Docker Image Size Report', '',
    size_df.to_markdown(index=False), '',
    '## Analysis',
    'The naive image uses the full python:3.11 base image and copies the entire project into the image, so it includes more operating-system packages and non-runtime files.',
    '',
    'The optimized image uses python:3.11-slim, installs dependencies in a builder stage, and copies only runtime dependencies plus the service source into the final image. I would use the optimized image in production because it is smaller, faster to pull, and has less unnecessary surface area.',
]
Path('reports/docker_size_report.md').write_text('\n'.join(report_lines) + '\n')
print('\nReport saved to reports/docker_size_report.md')

KeyError: "None of [Index(['Repository', 'Tag', 'Size'], dtype='object')] are in the [columns]"

### 3.4 Docker Compose

In [32]:
assert (PROJECT / 'docker-compose.yml').exists()
assert (PROJECT / '.env.example').exists()
assert '.env' in (PROJECT / '.gitignore').read_text()
print('Docker Compose, .env.example, and .gitignore are complete.')

Docker Compose, .env.example, and .gitignore are complete.


In [33]:
# Docker Compose smoke test
!docker compose up -d

import time, requests
time.sleep(8)  # wait for model to load from MLflow

health = requests.get('http://localhost:8000/health')
assert health.status_code == 200, f'Failed: {health.text}'
print('Docker Compose health check passed:', health.json())

!docker compose down

unable to get image 'qbc12-airbnb-serving:optimized': permission denied while trying to connect to the docker API at unix:///var/run/docker.sock


ConnectionError: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /health (Caused by NewConnectionError("HTTPConnection(host='localhost', port=8000): Failed to establish a new connection: [Errno 111] Connection refused"))

---
## Part 4 — Kubernetes Manifests

Kubernetes is the standard way to run containers in production at scale.

You do not need a real cluster for this homework. The deliverable is correct YAML files that a cluster could apply.

Key concepts you will use:

| Concept | What it does |
|---|---|
| **Pod** | Runs your container |
| **Deployment** | Manages multiple identical Pods, handles restarts |
| **Service** | Stable network endpoint that routes traffic to Pods |
| **Secret** | Stores sensitive values like passwords, not plaintext in YAML |
| **readinessProbe** | Tells Kubernetes when a Pod is ready to receive traffic |
| **resource limits** | Prevents one Pod from consuming all server memory |

### 4.1 Deployment

In [34]:
deployment_source = (PROJECT / 'k8s/deployment.yaml').read_text()
assert 'airbnb-serving-secret' in deployment_source
assert 'readinessProbe' in deployment_source
print('k8s/deployment.yaml is complete.')

k8s/deployment.yaml is complete.


### 4.2 Service

In [35]:
service_source = (PROJECT / 'k8s/service.yaml').read_text()
assert 'ClusterIP' in service_source
assert 'targetPort: 8000' in service_source
print('k8s/service.yaml is complete.')

k8s/service.yaml is complete.


### 4.3 Conceptual questions

Answer these in the markdown cell below. One or two sentences each is enough.

**TODO 4.3 — Answer here:**

**Q1.** We set `replicas: 2` instead of 1. What happens to traffic if one Pod crashes while replicas is 1 vs 2?

A: With `replicas: 1`, a crashed Pod causes downtime until Kubernetes replaces it. With `replicas: 2`, the Service can keep routing traffic to the remaining ready Pod while the failed one is replaced.

**Q2.** The `readinessProbe` has `initialDelaySeconds: 15`. Why do we need a delay specifically for this service?

A: The service loads a model from MLflow during startup, which can take several seconds. The delay prevents Kubernetes from sending traffic before the model is loaded and `/health` can respond.

**Q3.** Why do we store credentials in a Kubernetes Secret instead of writing them directly in `deployment.yaml`?

A: Secrets keep credentials out of the Deployment manifest and make it easier to manage sensitive values separately from application YAML. They are still not a replacement for proper cluster RBAC and secret management.

**Q4.** What is the difference between `ClusterIP` and `LoadBalancer` service types? When would you use each?

A: `ClusterIP` exposes the service only inside the cluster, which is appropriate for internal services. `LoadBalancer` asks the infrastructure provider for an external endpoint and is used when clients outside the cluster need direct access.

---
## Final Proof

If this cell fails, HW03 is not complete.

In [36]:
required_files = [
    'src/airbnb_serving/__init__.py',
    'src/airbnb_serving/schema.py',
    'src/airbnb_serving/predictor.py',
    'src/airbnb_serving/app.py',
    'pyproject.toml',
    'requirements.txt',
    'Dockerfile',
    'Dockerfile.naive',
    'docker-compose.yml',
    '.env.example',
    '.dockerignore',
    'k8s/deployment.yaml',
    'k8s/service.yaml',
    'reports/docker_size_report.md',
]

missing = [f for f in required_files if not (PROJECT / f).exists()]
assert not missing, f'Missing files:\n' + '\n'.join(missing)

# Check .env is gitignored
gitignore = (PROJECT / '.gitignore').read_text() if (PROJECT / '.gitignore').exists() else ''
assert '.env' in gitignore, '.env must be in .gitignore'

# Check Dockerfile does not copy .env
for df_name in ['Dockerfile', 'Dockerfile.naive']:
    content = (PROJECT / df_name).read_text()
    assert 'COPY .env' not in content, f'Do not copy .env in {df_name}'

# Check schema.py has actual content
schema_content = (PROJECT / 'src/airbnb_serving/schema.py').read_text()
assert 'BaseModel' in schema_content, 'schema.py must define Pydantic models'

# Check app.py has endpoints
app_content = (PROJECT / 'src/airbnb_serving/app.py').read_text()
assert '/health' in app_content, 'app.py must have /health endpoint'
assert '/predict' in app_content, 'app.py must have /predict endpoint'
assert 'batch' in app_content, 'app.py must have /predict/batch endpoint'

print('All required files present.')
print('No credential leaks detected.')
print('HW03 proof passed.')

All required files present.
No credential leaks detected.
HW03 proof passed.


## Screenshots required

Add these to the `screenshots/` folder before submitting:

- `screenshots/health_endpoint.png` — GET /health response
- `screenshots/predict_endpoint.png` — POST /predict response
- `screenshots/batch_endpoint.png` — POST /predict/batch response
- `screenshots/fastapi_docs.png` — auto-generated docs at /docs
- `screenshots/docker_image_sizes.png` — output of `docker images` showing both image sizes

## Commit

```bash
git add .
git commit -m "HW03 model serving and deployment"
git push
```